# Step 11 — Sequence BC (LSTM) Training

Trains a sequence behavior cloning model (LSTM) on `output/sim_trace_table.csv` using `sim_action` labels.
Saves model to `output/sequence_bc_lstm_model.pt`.

In [1]:
%pip install -q torch pandas numpy

In [3]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

ROOT = Path('.')
OUTPUT_DIR = ROOT / 'output'
TRACE_PATH = OUTPUT_DIR / 'sim_trace_table.csv'
ACTIONS_PATH = OUTPUT_DIR / 'valid_action_space.csv'
MODEL_PATH = OUTPUT_DIR / 'sequence_bc_lstm_model.pt'
REPORT_PATH = OUTPUT_DIR / 'sequence_bc_lstm_training_report.json'

if not TRACE_PATH.exists():
    raise FileNotFoundError(f'Missing trace file: {TRACE_PATH}')

df = pd.read_csv(TRACE_PATH)
required_cols = [
    'municipality', 'case_id', 'event_id', 'activity', 'prev_activity',
    'step_index', 'time_since_case_start_hours', 'time_since_prev_hours',
    'branch_label', 'branch_confidence', 'rework_count_activity', 'sim_action'
 ]
missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise ValueError(f'Missing required columns in sim_trace_table.csv: {missing}')

df = df.dropna(subset=['sim_action']).copy()

# Robust sim_action encoding: supports both integer ids and string action names.
sim_action_raw = df['sim_action'].astype(str).str.strip()
sim_action_num = pd.to_numeric(sim_action_raw, errors='coerce')

if sim_action_num.notna().all():
    df['sim_action'] = sim_action_num.astype(int)
else:
    if not ACTIONS_PATH.exists():
        raise FileNotFoundError(
            f'sim_action appears non-numeric, but action map is missing: {ACTIONS_PATH}'
        )
    actions_map_df = pd.read_csv(ACTIONS_PATH)
    needed = {'action_id', 'action_name'}
    if not needed.issubset(set(actions_map_df.columns)):
        raise ValueError(
            f'{ACTIONS_PATH} must contain columns: action_id and action_name'
        )

    action_name_to_id = {
        str(name).strip(): int(aid)
        for aid, name in zip(actions_map_df['action_id'], actions_map_df['action_name'])
    }
    mapped = sim_action_raw.map(action_name_to_id)
    if mapped.isna().any():
        unknown = sorted(sim_action_raw[mapped.isna()].unique().tolist())
        preview = unknown[:8]
        raise ValueError(
            f'Unmapped sim_action labels (showing up to 8): {preview}. '
            f'Please ensure names match {ACTIONS_PATH.name} action_name values.'
        )
    df['sim_action'] = mapped.astype(int)

df = df.sort_values(['municipality', 'case_id', 'event_id']).reset_index(drop=True)

cat_cols = ['activity', 'prev_activity', 'branch_label']
num_cols = ['municipality', 'step_index', 'time_since_case_start_hours', 'time_since_prev_hours', 'branch_confidence', 'rework_count_activity']

def build_vocab(series):
    vals = sorted([str(v) for v in series.fillna('unknown').unique().tolist()])
    return {v: i + 1 for i, v in enumerate(vals)}  # 0 = pad/unk

vocabs = {c: build_vocab(df[c]) for c in cat_cols}
action_values = sorted(df['sim_action'].unique().tolist())
action_to_idx = {a: i for i, a in enumerate(action_values)}
idx_to_action = {i: a for a, i in action_to_idx.items()}

for c in cat_cols:
    df[c] = df[c].fillna('unknown').astype(str).map(lambda v: vocabs[c].get(v, 0))

for c in num_cols:
    df[c] = pd.to_numeric(df[c], errors='coerce').fillna(0.0).astype(np.float32)

df['action_idx'] = df['sim_action'].map(action_to_idx).astype(int)

case_keys = df[['municipality', 'case_id']].drop_duplicates().reset_index(drop=True)
case_pairs = list(case_keys.itertuples(index=False, name=None))
rng = np.random.default_rng(42)
rng.shuffle(case_pairs)
split = int(0.8 * len(case_pairs))
train_pairs = set(case_pairs[:split])
valid_pairs = set(case_pairs[split:])

def collect_sequences(pairs):
    seqs = []
    grouped = df.groupby(['municipality', 'case_id'], sort=False)
    for key, g in grouped:
        if key not in pairs:
            continue
        seqs.append({
            'cat': g[cat_cols].to_numpy(dtype=np.int64),
            'num': g[num_cols].to_numpy(dtype=np.float32),
            'y': g['action_idx'].to_numpy(dtype=np.int64),
        })
    return seqs

train_seqs = collect_sequences(train_pairs)
valid_seqs = collect_sequences(valid_pairs)
print(f'Train sequences: {len(train_seqs)} | Valid sequences: {len(valid_seqs)} | Actions: {len(action_values)}')

Train sequences: 9600 | Valid sequences: 2400 | Actions: 6


In [4]:
class SequenceDataset(Dataset):
    def __init__(self, seqs):
        self.seqs = seqs

    def __len__(self):
        return len(self.seqs)

    def __getitem__(self, idx):
        return self.seqs[idx]

def collate(batch):
    bsz = len(batch)
    lens = [len(x['y']) for x in batch]
    tmax = max(lens)

    n_cat = batch[0]['cat'].shape[1]
    n_num = batch[0]['num'].shape[1]

    cat = torch.zeros((bsz, tmax, n_cat), dtype=torch.long)
    num = torch.zeros((bsz, tmax, n_num), dtype=torch.float32)
    y = torch.full((bsz, tmax), -100, dtype=torch.long)
    mask = torch.zeros((bsz, tmax), dtype=torch.bool)

    for i, row in enumerate(batch):
        t = len(row['y'])
        cat[i, :t] = torch.from_numpy(row['cat'])
        num[i, :t] = torch.from_numpy(row['num'])
        y[i, :t] = torch.from_numpy(row['y'])
        mask[i, :t] = True

    return cat, num, y, mask

class SequenceBCLSTM(nn.Module):
    def __init__(self, vocab_sizes, n_num, n_actions, emb_dim=16, hidden_size=128, n_layers=1):
        super().__init__()
        self.embs = nn.ModuleList([nn.Embedding(v + 1, emb_dim) for v in vocab_sizes])
        input_dim = len(vocab_sizes) * emb_dim + n_num
        self.lstm = nn.LSTM(input_dim, hidden_size, num_layers=n_layers, batch_first=True)
        self.head = nn.Linear(hidden_size, n_actions)

    def forward(self, cat, num):
        emb_list = []
        for i, emb in enumerate(self.embs):
            emb_list.append(emb(cat[:, :, i]))
        x = torch.cat(emb_list + [num], dim=-1)
        out, _ = self.lstm(x)
        return self.head(out)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
train_loader = DataLoader(SequenceDataset(train_seqs), batch_size=32, shuffle=True, collate_fn=collate)
valid_loader = DataLoader(SequenceDataset(valid_seqs), batch_size=32, shuffle=False, collate_fn=collate)

model = SequenceBCLSTM(
    vocab_sizes=[len(vocabs[c]) for c in cat_cols],
    n_num=len(num_cols),
    n_actions=len(action_values),
    emb_dim=16,
    hidden_size=128,
).to(device)

criterion = nn.CrossEntropyLoss(ignore_index=-100)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

def run_epoch(loader, train=False):
    model.train(mode=train)
    total_loss = 0.0
    total_ok = 0
    total_n = 0

    for cat, num, y, mask in loader:
        cat, num, y, mask = cat.to(device), num.to(device), y.to(device), mask.to(device)
        logits = model(cat, num)
        loss = criterion(logits.reshape(-1, logits.shape[-1]), y.reshape(-1))

        if train:
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

        with torch.no_grad():
            pred = logits.argmax(dim=-1)
            ok = ((pred == y) & mask).sum().item()
            n = mask.sum().item()

        total_loss += float(loss.item())
        total_ok += int(ok)
        total_n += int(n)

    mean_loss = total_loss / max(len(loader), 1)
    acc = total_ok / max(total_n, 1)
    return mean_loss, acc

history = []
n_epochs = 8
for epoch in range(1, n_epochs + 1):
    tr_loss, tr_acc = run_epoch(train_loader, train=True)
    va_loss, va_acc = run_epoch(valid_loader, train=False)
    history.append({
        'epoch': epoch,
        'train_loss': tr_loss,
        'train_acc': tr_acc,
        'valid_loss': va_loss,
        'valid_acc': va_acc,
    })
    print(f'Epoch {epoch:02d} | train_loss={tr_loss:.4f} train_acc={tr_acc:.4f} | valid_loss={va_loss:.4f} valid_acc={va_acc:.4f}')

payload = {
    'model_state_dict': model.state_dict(),
    'cat_cols': cat_cols,
    'num_cols': num_cols,
    'vocabs': vocabs,
    'action_to_idx': action_to_idx,
    'idx_to_action': idx_to_action,
    'emb_dim': 16,
    'hidden_size': 128,
}
torch.save(payload, MODEL_PATH)

REPORT_PATH.write_text(json.dumps({'history': history}, indent=2), encoding='utf-8')
print(f'Saved model: {MODEL_PATH}')
print(f'Saved report: {REPORT_PATH}')

Epoch 01 | train_loss=0.3100 train_acc=0.9186 | valid_loss=0.1224 valid_acc=0.9677
Epoch 02 | train_loss=0.0978 train_acc=0.9708 | valid_loss=0.1007 valid_acc=0.9669
Epoch 03 | train_loss=0.0753 train_acc=0.9756 | valid_loss=0.0805 valid_acc=0.9722
Epoch 04 | train_loss=0.0644 train_acc=0.9793 | valid_loss=0.0760 valid_acc=0.9735
Epoch 05 | train_loss=0.0591 train_acc=0.9800 | valid_loss=0.0626 valid_acc=0.9791
Epoch 06 | train_loss=0.0536 train_acc=0.9809 | valid_loss=0.0684 valid_acc=0.9772
Epoch 07 | train_loss=0.0515 train_acc=0.9814 | valid_loss=0.0402 valid_acc=0.9845
Epoch 08 | train_loss=0.0459 train_acc=0.9829 | valid_loss=0.0503 valid_acc=0.9810
Saved model: output/sequence_bc_lstm_model.pt
Saved report: output/sequence_bc_lstm_training_report.json


In [5]:
# Final block: export Step 11 artifacts from Colab runtime to Google Drive
from pathlib import Path
import shutil

LOCAL_OUTPUT_DIR = Path('output')
DRIVE_OUTPUT_DIR = Path('/content/drive/MyDrive/bureaucratic-workflow-analyzer/output/bc-lstm')

required_files = [
    LOCAL_OUTPUT_DIR / 'sequence_bc_lstm_model.pt',
    LOCAL_OUTPUT_DIR / 'sequence_bc_lstm_training_report.json',
]

IN_COLAB = False
try:
    import google.colab  # type: ignore
    from google.colab import drive  # type: ignore
    IN_COLAB = True
except Exception:
    IN_COLAB = False

if not IN_COLAB:
    print('Not running in Colab. Skipping Drive export.')
else:
    drive.mount('/content/drive', force_remount=False)
    DRIVE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    missing = [str(p) for p in required_files if not p.exists()]
    if missing:
        raise FileNotFoundError(
            f'Missing Step 11 output files: {missing}. Run training cell first.'
        )

    copied = 0
    updated = 0
    skipped = 0

    for src in required_files:
        dst = DRIVE_OUTPUT_DIR / src.name
        exists_before = dst.exists()
        should_copy = (
            (not exists_before)
            or (src.stat().st_size != dst.stat().st_size)
            or (src.stat().st_mtime > dst.stat().st_mtime)
        )

        if should_copy:
            shutil.copy2(src, dst)
            if exists_before:
                updated += 1
            else:
                copied += 1
        else:
            skipped += 1

    print('Step 11 artifact export complete.')
    print(f'Drive target: {DRIVE_OUTPUT_DIR}')
    print(f'Copied new: {copied} | Updated: {updated} | Skipped unchanged: {skipped}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Step 11 artifact export complete.
Drive target: /content/drive/MyDrive/bureaucratic-workflow-analyzer/output/bc-lstm
Copied new: 2 | Updated: 0 | Skipped unchanged: 0
